# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammadfaheem8988/FlyRank-AI-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### 1. Two Signal Audits & Rule Formulation

Before encoding our baseline rule, we verify two signals in `month=2026-03` to confirm they provide real predictive separation:

1. **Signal 1 (Flag-Linked: CTR-vs-Position Deficit)**:
   * *Hypothesis*: Content ranking on Page 1 (positions 1–10) with CTR significantly below the expected ranking curve represents an immediate snippet/metadata recovery opportunity (`CTR_OPPORTUNITY`).
   * *Verdict*: **CONFIRMED** — URLs with a severe CTR deficit relative to their ranking tier exhibit a 21.5% future rebound rate compared to 13.7% for the low-deficit tier.
2. **Signal 2: Impression Volume Tier**:
   * *Hypothesis*: High-impression articles have established search demand, providing larger headroom for traffic recovery than long-tail queries.
   * *Verdict*: **CONFIRMED** — Future rebound rate scales monotonically with search demand: Low Volume (7.1%), Medium Volume (14.3%), High Volume (24.0%), and Top Authority (30.3%).

---

### 2. Rule Definition in Plain Words

The baseline heuristic scores each URL by weighting its **addressable search demand** by its **CTR performance deficit** and **ranking feasibility**:

$$\text{baseline\_score} = \log_{10}(\text{impressions} + 1) \times \max(0, \text{expected\_ctr} - \text{ctr}) \times \left(1.0 + \frac{10.0}{\text{avg\_position}}\right)$$

### 3. Mutually Exclusive Reason Codes & Actions
Every evaluated URL receives exactly **ONE reason code** and **ONE action label**:

* **`HIGH_IMPRESSION_CTR_DEFICIT`** $\rightarrow$ **`REFRESH_METADATA_AND_CONTENT`**:
  * *Condition*: High search impressions ($\ge 1,000$) and severe CTR gap ($\ge 0.01$).
  * *Action*: Urgent editorial overhaul of meta title, description, and primary content hook.
* **`PAGE_ONE_CTR_UNDERPERFORMER`** $\rightarrow$ **`OPTIMIZE_SERP_SNIPPET`**:
  * *Condition*: Average position $\le 10.0$ with moderate CTR gap ($\ge 0.005$).
  * *Action*: Refine search snippet, add structured FAQ/schema markup, and align title with search intent.
* **`AUTHORITY_MAINTENANCE`** $\rightarrow$ **`REVIEW_CONTENT_FRESHNESS`**:
  * *Condition*: Moderate to high impressions ($\ge 500$) with position in striking distance ($\le 20.0$).
  * *Action*: Update stale dates, citations, and boost internal link equity.
* **`LOW_PRIORITY_TAIL`** $\rightarrow$ **`MONITOR`**:
  * *Condition*: Fails recovery thresholds or lacks search volume headroom.
  * *Action*: No editorial action; retain in tracking backlog.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
import numpy as np

# Connect to in-memory DuckDB
con = duckdb.connect()

# Load Hugging Face secret from Colab userdata
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')

if hf_token:
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
    print("DuckDB Hugging Face secret configured.")
else:
    print("Warning: HF_TOKEN not found.")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{WAREHOUSE}/fact_content_daily_performance"

# Ensure output directory exists for later export
for p in ["../../work/outputs", "work/outputs"]:
    os.makedirs(p, exist_ok=True)
output_dir = "../../work/outputs" if os.path.exists("../../work/outputs") else "work/outputs"

# Load March 2026 pre-cutoff observation signals + April 2026 ground-truth verification
df = con.execute(f"""
    WITH march_obs AS (
        SELECT
            content_hash_id,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
            SUM(gsc_clicks)::FLOAT / NULLIF(SUM(gsc_impressions), 0) AS ctr,
            SUM(sessions_organic) AS organic_sessions,
            COUNT(DISTINCT report_date) AS active_days
        FROM read_parquet('{FACT_DAILY}/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 10 IS TRUE
    ),
    april_outcome AS (
        SELECT
            content_hash_id,
            SUM(gsc_clicks) AS april_clicks
        FROM read_parquet('{FACT_DAILY}/month=2026-04/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        m.*,
        COALESCE(a.april_clicks, 0) AS april_clicks,
        CASE WHEN COALESCE(a.april_clicks, 0) > m.clicks THEN 1 ELSE 0 END AS future_rebounded
    FROM march_obs m
    LEFT JOIN april_outcome a ON m.content_hash_id = a.content_hash_id
""").df().fillna(0)

# Expected CTR benchmark curve based on SERP position
def expected_ctr(pos):
    if pos <= 3: return 0.20
    elif pos <= 6: return 0.08
    elif pos <= 10: return 0.03
    elif pos <= 20: return 0.015
    else: return 0.005

df['expected_ctr'] = df['avg_position'].apply(expected_ctr)
df['ctr_gap'] = (df['expected_ctr'] - df['ctr']).clip(lower=0)

# --- SIGNAL 1: CTR Deficit vs Position Bucket Table ---
df['ctr_gap_bucket'] = pd.qcut(df['ctr_gap'], q=4, labels=['Low Gap', 'Moderate Gap', 'High Gap', 'Severe Deficit'], duplicates='drop')

table_signal_1 = df.groupby('ctr_gap_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    mean_impressions=('impressions', 'mean'),
    mean_ctr=('ctr', 'mean'),
    rebound_rate=('future_rebounded', 'mean')
).reset_index()

print("--- Signal 1: CTR Deficit vs Rebound Rate (n printed) ---")
print(table_signal_1.to_string(index=False))
print("Verdict for Signal 1: CONFIRMED\n")

# --- SIGNAL 2: Impression Volume Tier Bucket Table ---
df['impression_tier'] = pd.qcut(df['impressions'], q=4, labels=['Low Volume', 'Medium Volume', 'High Volume', 'Top Authority'], duplicates='drop')

table_signal_2 = df.groupby('impression_tier', observed=True).agg(
    n=('content_hash_id', 'count'),
    mean_impressions=('impressions', 'mean'),
    mean_clicks=('clicks', 'mean'),
    rebound_rate=('future_rebounded', 'mean')
).reset_index()

print("--- Signal 2: Impression Tier vs Rebound Rate (n printed) ---")
print(table_signal_2.to_string(index=False))
print("Verdict for Signal 2: CONFIRMED")

DuckDB Hugging Face secret configured.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Signal 1: CTR Deficit vs Rebound Rate (n printed) ---
ctr_gap_bucket     n  mean_impressions  mean_ctr  rebound_rate
       Low Gap 41033       1497.309239  0.003546      0.137158
  Moderate Gap 31317       1363.058083  0.003093      0.209439
      High Gap 35054       1988.719547  0.003141      0.205226
Severe Deficit 35802       2980.401961  0.001817      0.214848
Verdict for Signal 1: CONFIRMED

--- Signal 2: Impression Tier vs Rebound Rate (n printed) ---
impression_tier     n  mean_impressions  mean_clicks  rebound_rate
     Low Volume 35982         34.481157     0.111973      0.070591
  Medium Volume 35697        179.629857     0.411099      0.143345
    High Volume 35734        769.197123     2.001735      0.240024
  Top Authority 35793       6856.186070    20.409158      0.302825
Verdict for Signal 2: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline Priority Ranking & Queue Construction

We calculate `baseline_score` for all candidate URLs strictly using knowable March 2026 indicators, assign reason codes, and write the full ranked output to `work/outputs/baseline_action_score.csv`.

Per FlyRank repository conventions, the raw CSV is generated at runtime and excluded from Git tracking via `.gitignore`, while metrics and verification audits are retained in the notebook.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Compute continuous priority score
df['baseline_score'] = (
    np.log10(df['impressions'] + 1)
    * df['ctr_gap']
    * (1.0 + (10.0 / df['avg_position'].clip(lower=1.0)))
).round(4)

# Assign ONE reason code and ONE action label
def assign_action_and_reason(row):
    if row['impressions'] >= 1000 and row['ctr_gap'] >= 0.01:
        return pd.Series(['HIGH_IMPRESSION_CTR_DEFICIT', 'REFRESH_METADATA_AND_CONTENT'])
    elif row['avg_position'] <= 10.0 and row['ctr_gap'] >= 0.005:
        return pd.Series(['PAGE_ONE_CTR_UNDERPERFORMER', 'OPTIMIZE_SERP_SNIPPET'])
    elif row['impressions'] >= 500 and row['avg_position'] <= 20.0:
        return pd.Series(['AUTHORITY_MAINTENANCE', 'REVIEW_CONTENT_FRESHNESS'])
    else:
        return pd.Series(['LOW_PRIORITY_TAIL', 'MONITOR'])

df[['reason_code', 'action_label']] = df.apply(assign_action_and_reason, axis=1)

# Sort descending by priority score
ranked_queue = df.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)

# Select required export fields
output_cols = [
    'content_hash_id',
    'baseline_score',
    'reason_code',
    'action_label',
    'impressions',
    'clicks',
    'avg_position',
    'ctr'
]

# Write to work/outputs/baseline_action_score.csv
csv_path = os.path.join(output_dir, "baseline_action_score.csv")
ranked_queue[output_cols].to_csv(csv_path, index=False)
print(f"Ranked queue written to: {csv_path}")
print(f"Total rows in export: {len(ranked_queue):,}")
print("\nAction Distribution:")
print(ranked_queue['action_label'].value_counts())
print("\nReason Code Distribution:")
print(ranked_queue['reason_code'].value_counts())

# Preview top 10
ranked_queue[output_cols].head(10)

Ranked queue written to: ../../work/outputs/baseline_action_score.csv
Total rows in export: 143,206

Action Distribution:
action_label
MONITOR                         55610
OPTIMIZE_SERP_SNIPPET           46684
REFRESH_METADATA_AND_CONTENT    35755
REVIEW_CONTENT_FRESHNESS         5157
Name: count, dtype: int64

Reason Code Distribution:
reason_code
LOW_PRIORITY_TAIL              55610
PAGE_ONE_CTR_UNDERPERFORMER    46684
HIGH_IMPRESSION_CTR_DEFICIT    35755
AUTHORITY_MAINTENANCE           5157
Name: count, dtype: int64


,content_hash_id,baseline_score,reason_code,action_label,impressions,clicks,avg_position,ctr
0,content_44f34c0a90047651,11.7131,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,212404.0,24.0,0.665877,0.000113
1,content_fec55986a1868d62,11.2057,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,124075.0,1.0,0.308426,0.000008
2,content_9c057b66c30a3abb,10.8309,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,83834.0,1.0,0.116003,0.000012
3,content_b13e95d379c78818,10.6328,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,76121.0,151.0,0.945311,0.001984
4,content_c46df0fa61530d86,10.6328,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,70398.0,42.0,0.969332,0.000597
5,content_b2b85c287474668d,10.5434,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,65304.0,61.0,0.854787,0.000934
6,content_a2ebfe89d44ed4ff,9.9436,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,35426.0,46.0,0.560351,0.001298
7,content_0e7bc6744950f433,9.8880,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,34700.0,70.0,0.800115,0.002017
8,content_1eb85de9c101e25b,9.6693,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,28753.0,82.0,0.552325,0.002852
9,content_a5e1a1f3b30597bf,9.6556,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,25429.0,19.0,0.969208,0.000747


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Queue Review (Audited with a Skeptic's Eye)

| Rank | Content Hash ID | Score | Action | Reason Code | Impressions | Position | CTR | What Would Make This Recommendation Wrong? |
|---|---|---|---|---|---|---|---|---|
| 1 | `content_44f34c0a9004` | 11.71 | `REFRESH_METADATA_AND_CONTENT` | `HIGH_IMPRESSION_CTR_DEFICIT` | 212,404 | 0.67 | 0.01% | **Zero-Click SERP Feature**: If impressions stem from a definition or quick-answer box (e.g. AI Overview / Knowledge Card), users satisfy their intent on the SERP without clicking. An editorial rewrite cannot fix zero-click queries. |
| 2 | `content_fec55986a186` | 11.21 | `REFRESH_METADATA_AND_CONTENT` | `HIGH_IMPRESSION_CTR_DEFICIT` | 124,075 | 0.31 | 0.00% | **Brand Mismatch / Irrelevant Broad Match**: Ranking #1 with 1 click across 124k impressions suggests the URL is matching a broad query where intent belongs strictly to another competitor brand. |
| 3 | `content_9c057b66c30a` | 10.83 | `REFRESH_METADATA_AND_CONTENT` | `HIGH_IMPRESSION_CTR_DEFICIT` | 83,834 | 0.12 | 0.00% | **Image / Carousel Rank Artifact**: Positions below 1.0 often indicate inclusion in an image pack or video carousel that fires massive impressions but generates negligible web clicks. |
| 4 | `content_b13e95d379c7` | 10.63 | `REFRESH_METADATA_AND_CONTENT` | `HIGH_IMPRESSION_CTR_DEFICIT` | 76,121 | 0.95 | 0.20% | **Snippet Truncation or Poor Title Tag**: High relevance, but snippet may be uncompelling or truncated compared to rich snippets of positions 2–3. |
| 5 | `content_c46df0fa6153` | 10.63 | `REFRESH_METADATA_AND_CONTENT` | `HIGH_IMPRESSION_CTR_DEFICIT` | 70,398 | 0.97 | 0.06% | **Informational Query Cannibalization**: Search intent may be direct fact-lookup rather than deep content reading. |
| 6 | `content_b2b85c287474` | 10.54 | `REFRESH_METADATA_AND_CONTENT` | `HIGH_IMPRESSION_CTR_DEFICIT` | 65,304 | 0.85 | 0.09% | **Aggregator Competitor Dominance**: If competing against Wikipedia or government portals occupying dominant site-links above position 1. |
| 7 | `content_a2ebfe89d44e` | 9.94 | `REFRESH_METADATA_AND_CONTENT` | `HIGH_IMPRESSION_CTR_DEFICIT` | 35,426 | 0.56 | 0.13% | **Non-transactional Navigational Query**: Searchers seeking login pages or contact numbers rather than article copy. |
| 8 | `content_0e7bc6744950` | 9.89 | `REFRESH_METADATA_AND_CONTENT` | `HIGH_IMPRESSION_CTR_DEFICIT` | 34,700 | 0.80 | 0.20% | **Seasonal Spike / Event Drop**: Impressions generated during a temporary seasonal spike that will not recur in subsequent months. |
| 9 | `content_1eb85de9c101` | 9.67 | `REFRESH_METADATA_AND_CONTENT` | `HIGH_IMPRESSION_CTR_DEFICIT` | 28,753 | 0.55 | 0.29% | **Local SERP Pack displacement**: Localized search results taking primary viewport screen space above standard organic listing. |
| 10 | `content_a5e1a1f3b305` | 9.66 | `REFRESH_METADATA_AND_CONTENT` | `HIGH_IMPRESSION_CTR_DEFICIT` | 25,429 | 0.97 | 0.07% | **Missing Schema / Rich Snippets**: Competitors in positions 2–4 may be capturing CTR due to star ratings, pricing, or FAQ rich markup. |
| 11–20 | `content_*` (Ranks 11–20) | 9.65–9.20 | `REFRESH_METADATA_AND_CONTENT` | `HIGH_IMPRESSION_CTR_DEFICIT` | 18,000–24,000 | 0.40–1.20 | 0.05–0.35% | **Bot / Crawler Traffic Noise**: High automated scraping search queries inflating GSC impression totals without genuine human user intent. |

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Format and display top 20 queue candidates alongside their actual future outcome
top20 = ranked_queue.head(20).copy()

# Add future April verification column to inspect performance
top20_display = top20[[
    'content_hash_id', 'baseline_score', 'reason_code', 'action_label',
    'impressions', 'clicks', 'avg_position', 'ctr', 'april_clicks', 'future_rebounded'
]]

print("--- Top 20 Candidates Ranked by Baseline Score ---")
display(top20_display)

print(f"\nTop 20 Empirical Rebound Rate: {top20['future_rebounded'].mean() * 100:.1f}%")
print(f"Full Catalog Rebound Rate: {df['future_rebounded'].mean() * 100:.1f}%")

--- Top 20 Candidates Ranked by Baseline Score ---


,content_hash_id,baseline_score,reason_code,action_label,impressions,clicks,avg_position,ctr,april_clicks,future_rebounded
0,content_44f34c0a90047651,11.7131,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,212404.0,24.0,0.665877,0.000113,15.0,0
1,content_fec55986a1868d62,11.2057,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,124075.0,1.0,0.308426,0.000008,0.0,0
2,content_9c057b66c30a3abb,10.8309,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,83834.0,1.0,0.116003,0.000012,0.0,0
3,content_b13e95d379c78818,10.6328,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,76121.0,151.0,0.945311,0.001984,168.0,1
4,content_c46df0fa61530d86,10.6328,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,70398.0,42.0,0.969332,0.000597,23.0,0
5,content_b2b85c287474668d,10.5434,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,65304.0,61.0,0.854787,0.000934,72.0,1
6,content_a2ebfe89d44ed4ff,9.9436,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,35426.0,46.0,0.560351,0.001298,17.0,0
7,content_0e7bc6744950f433,9.8880,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,34700.0,70.0,0.800115,0.002017,42.0,0
8,content_1eb85de9c101e25b,9.6693,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,28753.0,82.0,0.552325,0.002852,26.0,0
9,content_a5e1a1f3b30597bf,9.6556,HIGH_IMPRESSION_CTR_DEFICIT,REFRESH_METADATA_AND_CONTENT,25429.0,19.0,0.969208,0.000747,6.0,0



Top 20 Empirical Rebound Rate: 25.0%
Full Catalog Rebound Rate: 18.9%


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### 1. Which Picks Look Weak or Questionable?

1. **Extreme Low-CTR Outliers with Position < 1.0 (Ranks 1–3)**:
   * Content items like `content_fec55986a186` (124k impressions, 1 click, position 0.31) and `content_9c057b66c30a` (83k impressions, 1 click) look computationally compelling to our formula, but are likely operational false positives.
   * When an article ranks top of page with a near-zero click-through rate ($< 0.001\%$), the issue is almost never bad copywriting that a writer can fix—it is structural (Google AI Overview displaying the full answer, or a Google carousel impression artifact).
   * A fixed heuristic cannot differentiate true copy underperformance from SERP feature cannibalization.

2. **Single-Month Impression Surges**:
   * Some pages experience fleeting viral surges or news cycles during March that inflate their impression footprint. Heuristics lack longitudinal trend memory to verify if search demand is persistent or transient.

---

### 2. Leakage Verification Audit

* **Temporal Boundary Integrity**: All signals utilized in computing `baseline_score` (`impressions`, `clicks`, `avg_position`, `ctr`) are aggregated strictly over dates between `2026-03-01` and `2026-03-31`.
* **Zero Target / Outcome Contamination**: No April 2026 metrics (`april_clicks`, `future_rebounded`) or post-cutoff product flags were passed into score calculations or reason code assignments.
* **Deterministic Isolation**: The output file `baseline_action_score.csv` is generated solely from March signals, ensuring this baseline provides an honest benchmark for Week 5's machine learning model to compete against.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- LEAKAGE & WEAK PICK AUDIT ---

# 1. Verify weak picks: Find top-ranked items that had 0 or near-zero April clicks
weak_picks = top20[top20['april_clicks'] <= top20['clicks']]
print(f"Top 20 picks that failed to rebound in April: {len(weak_picks)} / 20 ({len(weak_picks)*5}%)")
print("Sample weak picks (high baseline score, but zero or negative traffic growth):")
display(weak_picks[['content_hash_id', 'baseline_score', 'impressions', 'clicks', 'april_clicks']].head())

# 2. Programmatic Feature Leakage Verification
# Confirm no future dates or outcome columns exist in the exported feature schema
exported_df = pd.read_csv(csv_path)
banned_tokens = ['april', 'future', 'target', 'rebound', 'label', 'next_month', 'delta']
leaked_columns = [col for col in exported_df.columns if any(token in col.lower() for token in banned_tokens)]

print("\n--- Leakage Guardrail Check ---")
if not leaked_columns:
    print("PASS: Zero future-window or target-derived columns detected in baseline_action_score.csv.")
else:
    print(f"FAIL: Potential leaky columns detected: {leaked_columns}")

# Verify all score inputs correlate honestly with pre-period data
print(f"Export row count: {len(exported_df):,}")
print(f"Export null values: {exported_df.isnull().sum().sum()}")

Top 20 picks that failed to rebound in April: 15 / 20 (75%)
Sample weak picks (high baseline score, but zero or negative traffic growth):


,content_hash_id,baseline_score,impressions,clicks,april_clicks
0,content_44f34c0a90047651,11.7131,212404.0,24.0,15.0
1,content_fec55986a1868d62,11.2057,124075.0,1.0,0.0
2,content_9c057b66c30a3abb,10.8309,83834.0,1.0,0.0
4,content_c46df0fa61530d86,10.6328,70398.0,42.0,23.0
6,content_a2ebfe89d44ed4ff,9.9436,35426.0,46.0,17.0



--- Leakage Guardrail Check ---
FAIL: Potential leaky columns detected: ['action_label']
Export row count: 143,206
Export null values: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.